In [19]:
import numpy as np
import torch
import pandas as pd
from helper import Autoencoder, load_data, train, save_params

In [20]:
known_strengths = {'null':10,'N4': 0.0, 'Q4': 1.3340727612197436, 'Q7': 2.428134794028789, 'T4': 1.9599578912997808, 'V4': 3.2307473950102388, 'G4': 4.514668716435611, 'E1': 5.21564553829087, 'A2': 0.43209185328878835, 'Y3': 0.8603530802315512}
train_strength = True

In [21]:
# Parameters for the autoencoder that can be changed

ignore_indexes=['Event','Replicate'] #ensure at least Event is ignored so that it can group properly since it is unique to each cell
group_size=100
batch_size=100
reconstruction_weight=1
strength_weight=0.0001
num_epochs=15
embedding_size=2
autoencoder_hidden_sizes=[250,200]
train_test_split=0.8
model_inputs = ['CD126', 'CD19', 'CD2', 'CD25', 'CD27', 'CD38', 'CD4', 'CD44', 'CD45',
       'CD45RA', 'CD5', 'CD62L', 'CD86', 'CD8a', 'CX3CR1', 'CXCR6', 'FSC-A',
       'Granzyme B', 'ICOS', 'IRF8', 'MHC-II', 'OX40', 'PD-L1',
       'Proliferation', 'SSC-A', 'TBet']
# Try without CD38 (combined dataset has no CD38)
# model_inputs = ['CD126', 'CD19', 'CD2', 'CD25', 'CD27', 'CD4', 'CD44', 'CD45',
#        'CD45RA', 'CD5', 'CD62L', 'CD86', 'CD8a', 'CX3CR1', 'CXCR6', 'FSC-A',
#        'Granzyme B', 'ICOS', 'IRF8', 'MHC-II', 'OX40', 'PD-L1',
#        'Proliferation', 'SSC-A', 'TBet']
model_path = "autoencoder_test.pt"
data_path = "../../initialSingleCellDf-channel-20220916-MW_018-001.h5"
# data_path = "../../initialSingleCellDf-channel-20220926-MW_020.h5"

def get_strength(labels,data_index_names):
       if 'Peptide' not in data_index_names:
              print("No peptide column found")
              return -1
       antigen_column = list(data_index_names).index('Peptide')
       antigen = labels[antigen_column]
       if antigen in known_strengths:
              return known_strengths[antigen]
       else:
              print("Antigen not found: ",antigen)
              return -1

In [22]:
data = pd.read_hdf(data_path, key="df")
# Add any filters here to remove data that you don't want the model to be trained on
data = data.loc[(data.index.get_level_values('CellType') == 'OT-1') & (data.index.get_level_values('Peptide') != 'N4')]# & ((data.index.get_level_values('Time') >= 40) | (data.index.get_level_values('Peptide') == 'null'))]#& (data.index.get_level_values('Peptide') != 'T4')]

In [23]:
dataset, index_order, data_index_names, data_columns, data_labels,missing_columns = load_data(data,model_inputs,ignore_indexes,group_size,transform='normalize')
print("Missing columns: ",missing_columns)

Missing columns:  []


In [24]:
# add to the dataset the known strengths for each sample

def add_column_to_label(dataset,new_column):
    data = []
    labels = []
    for i in range(len(dataset)):
        new_labels = np.append(dataset[i][1],new_column[i])
        data.append(np.array(dataset[i][0]))
        labels.append(new_labels)
    data = np.array(data)
    labels = np.array(labels, dtype=np.float32)  # Convert labels to float32
    new_dataset = torch.utils.data.TensorDataset(torch.tensor(data),torch.tensor(labels))
    return new_dataset, len(labels[0])-1

strengths = [get_strength(labels,data_index_names) for labels in data_labels]
new_dataset, strength_index = add_column_to_label(dataset,strengths)


In [25]:
train_size = int(train_test_split * len(new_dataset))
test_size = len(new_dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(new_dataset, [train_size, test_size])
train_loader = torch.utils.data.DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset,batch_size=batch_size,shuffle=True)

In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model = Autoencoder(len(model_inputs),autoencoder_hidden_sizes,embedding_size).to(device)

In [27]:
train(model,train_loader,test_loader,reconstruction_weight,strength_weight,strength_index,device,num_epochs=num_epochs,train_strength=train_strength)
torch.save(model.state_dict(), model_path)
save_params(model_inputs,group_size,batch_size,embedding_size,autoencoder_hidden_sizes,model_path)

epoch [1/15], train loss:0.167403 val loss:0.120164 val reconstruction loss 0.119788 val strength loss 3.755755
epoch [2/15], train loss:0.110785 val loss:0.104814 val reconstruction loss 0.104438 val strength loss 3.756995
epoch [3/15], train loss:0.101375 val loss:0.097855 val reconstruction loss 0.097469 val strength loss 3.867449
epoch [4/15], train loss:0.095237 val loss:0.094896 val reconstruction loss 0.094506 val strength loss 3.893357
epoch [5/15], train loss:0.090828 val loss:0.087565 val reconstruction loss 0.087173 val strength loss 3.920388
epoch [6/15], train loss:0.087137 val loss:0.087327 val reconstruction loss 0.086930 val strength loss 3.978629
epoch [7/15], train loss:0.084547 val loss:0.084432 val reconstruction loss 0.084028 val strength loss 4.036248
epoch [8/15], train loss:0.082101 val loss:0.082771 val reconstruction loss 0.082368 val strength loss 4.027324
epoch [9/15], train loss:0.080329 val loss:0.080500 val reconstruction loss 0.080093 val strength loss 4